# 04 ML PyCaret Suite
Start implementation for multi-track ML using cleaned Olist datasets.

## 1) Set Up Environment and Dependencies
Import required libraries, set reproducibility seed, and verify key package versions.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any
import random

import numpy as np
import pandas as pd
import pycaret

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("pycaret", pycaret.__version__)
print("pandas", pd.__version__)

pycaret 3.3.2
pandas 2.1.4


## 2) Define Configuration and Constants
Centralize paths and run settings used by all tracks.

In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_ORDER_PATH = PROCESSED_DIR / "feature_mart_order_level.parquet"
FEATURE_CUSTOMER_PATH = PROCESSED_DIR / "feature_mart_customer_level.parquet"

print(PROCESSED_DIR)
print(FEATURE_ORDER_PATH.exists(), FEATURE_CUSTOMER_PATH.exists())

c:\Users\sayee\Documents\olist-ai-analyst\data\processed
True True


## 3) Implement Core Data Structures
Define typed config and result entities for track execution.

In [3]:
@dataclass
class TrackConfig:
    name: str
    task_type: str
    target_column: str | None
    input_path: Path


@dataclass
class TrackResult:
    name: str
    status: str
    details: str
    metric_value: float | None = None

## 4) Implement Main Processing Functions
Create reusable load, validate, and summary functions.

In [3]:
def load_dataframe(path: Path) -> pd.DataFrame:
    """Load a parquet file and raise a clear error if missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing expected feature mart: {path}")
    return pd.read_parquet(path)


def ensure_columns(df: pd.DataFrame, required: list[str]) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def summarize_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    return df[cols].describe().T

## 5) Run a Minimal End-to-End Example
Load engineered features and produce a concrete metrics artifact.

In [4]:
order_df = load_dataframe(FEATURE_ORDER_PATH)
customer_df = load_dataframe(FEATURE_CUSTOMER_PATH)

required_order_cols = ["item_count", "total_price", "total_freight", "total_payment"]
ensure_columns(order_df, required_order_cols)

summary = summarize_numeric(order_df, required_order_cols)
display(summary)

summary_out = REPORTS_DIR / "ml_track_input_summary.csv"
summary.to_csv(summary_out)
print(f"Wrote {summary_out}")

,count,mean,std,min,25%,50%,75%,max
item_count,98666.0,1.141731,0.538452,1.00,1.00,1.00,1.00,21.00
total_price,98666.0,137.754076,210.645145,0.85,45.90,86.90,149.90,13440.00
total_freight,98666.0,22.823562,21.650909,0.00,13.85,17.17,24.04,1794.96
total_payment,99440.0,160.990267,221.951257,0.00,62.01,105.29,176.97,13664.08


Wrote c:\Users\sayee\Documents\olist-ai-analyst\reports\ml_track_input_summary.csv


## 6) Add Basic Validation Tests
Use simple assertions to verify critical assumptions before full PyCaret training loops.

In [5]:
assert not order_df.empty, "order_df must not be empty"
assert not customer_df.empty, "customer_df must not be empty"
assert order_df["total_payment"].isna().mean() < 0.2, "Too many nulls in total_payment"
assert order_df["item_count"].dropna().ge(0).all(), "item_count contains negative values"

print("Basic validation tests passed.")

Basic validation tests passed.


## 7) Prepare Modeling Datasets
Create track-specific datasets from cleaned and engineered feature marts.

In [6]:
orders_clean = load_dataframe(PROCESSED_DIR / "olist_orders_clean.parquet")

class_df = order_df[["item_count", "total_price", "total_freight", "total_payment"]].dropna().copy()
class_threshold = class_df["total_payment"].median()
class_df["target_high_value_order"] = (class_df["total_payment"] > class_threshold).astype(int)
class_df = class_df.drop(columns=["total_payment"])

reg_df = order_df[["item_count", "total_price", "total_freight", "total_payment"]].dropna().copy()

cluster_df = customer_df[["total_orders", "total_spend", "avg_order_value"]].dropna().copy()

anomaly_df = order_df[["item_count", "total_price", "total_freight", "total_payment"]].dropna().copy()

ts_df = (
    orders_clean.dropna(subset=["order_purchase_timestamp"])
    .assign(order_date=lambda d: pd.to_datetime(d["order_purchase_timestamp"]).dt.date)
    .groupby("order_date", as_index=False)
    .size()
    .rename(columns={"size": "daily_orders"})
)
ts_df["order_date"] = pd.to_datetime(ts_df["order_date"])

display(pd.DataFrame({
    "dataset": ["classification", "regression", "clustering", "anomaly", "time_series"],
    "rows": [len(class_df), len(reg_df), len(cluster_df), len(anomaly_df), len(ts_df)],
}))

,dataset,rows
0,classification,98665
1,regression,98665
2,clustering,96095
3,anomaly,98665
4,time_series,634


## 8) Run PyCaret ML Tracks
Train one baseline model per track and persist artifacts and metrics.

In [7]:
from pycaret.classification import setup as cls_setup, compare_models as cls_compare_models, pull as cls_pull, save_model as cls_save_model, predict_model as cls_predict_model
from pycaret.regression import setup as reg_setup, compare_models as reg_compare_models, pull as reg_pull, save_model as reg_save_model, predict_model as reg_predict_model
from pycaret.clustering import setup as clu_setup, create_model as clu_create_model, assign_model as clu_assign_model, pull as clu_pull, save_model as clu_save_model
from pycaret.anomaly import setup as anm_setup, create_model as anm_create_model, assign_model as anm_assign_model, save_model as anm_save_model
from pycaret.time_series import TSForecastingExperiment

track_rows: list[dict[str, Any]] = []

# Classification: high-value order flag
cls_setup(data=class_df, target="target_high_value_order", session_id=SEED, fold=3, html=False, verbose=False)
cls_model = cls_compare_models(include=["lr", "rf", "lightgbm"], turbo=True)
cls_metrics = cls_pull().copy()
cls_pred = cls_predict_model(cls_model)
cls_save_model(cls_model, str(MODELS_DIR / "classification_high_value_best"))
cls_pred.to_csv(REPORTS_DIR / "classification_predictions.csv", index=False)
cls_metrics.to_csv(REPORTS_DIR / "classification_metrics.csv", index=False)
track_rows.append({
    "track": "classification",
    "status": "ok",
    "primary_metric": "Accuracy",
    "primary_metric_value": float(cls_metrics.iloc[0]["Accuracy"]) if "Accuracy" in cls_metrics.columns else None,
    "model_artifact": str(MODELS_DIR / "classification_high_value_best.pkl"),
})

# Regression: total payment
reg_setup(data=reg_df, target="total_payment", session_id=SEED, fold=3, html=False, verbose=False)
reg_model = reg_compare_models(include=["lr", "rf", "lightgbm"], turbo=True)
reg_metrics = reg_pull().copy()
reg_pred = reg_predict_model(reg_model)
reg_save_model(reg_model, str(MODELS_DIR / "regression_total_payment_best"))
reg_pred.to_csv(REPORTS_DIR / "regression_predictions.csv", index=False)
reg_metrics.to_csv(REPORTS_DIR / "regression_metrics.csv", index=False)
track_rows.append({
    "track": "regression",
    "status": "ok",
    "primary_metric": "MAE",
    "primary_metric_value": float(reg_metrics.iloc[0]["MAE"]) if "MAE" in reg_metrics.columns else None,
    "model_artifact": str(MODELS_DIR / "regression_total_payment_best.pkl"),
})

# Clustering: customer segmentation
clu_setup(data=cluster_df, session_id=SEED, html=False, verbose=False, normalize=True)
clu_model = clu_create_model("kmeans")
clu_metrics = clu_pull().copy()
clu_assigned = clu_assign_model(clu_model)
clu_save_model(clu_model, str(MODELS_DIR / "clustering_customer_kmeans"))
clu_assigned.to_csv(REPORTS_DIR / "clustering_assignments.csv", index=False)
clu_metrics.to_csv(REPORTS_DIR / "clustering_metrics.csv", index=False)
track_rows.append({
    "track": "clustering",
    "status": "ok",
    "primary_metric": "Silhouette",
    "primary_metric_value": float(clu_metrics.iloc[0]["Silhouette"]) if "Silhouette" in clu_metrics.columns else None,
    "model_artifact": str(MODELS_DIR / "clustering_customer_kmeans.pkl"),
})

# Anomaly detection: unusual orders
anm_setup(data=anomaly_df, session_id=SEED, html=False, verbose=False, normalize=True)
anm_model = anm_create_model("iforest")
anm_assigned = anm_assign_model(anm_model)
anm_save_model(anm_model, str(MODELS_DIR / "anomaly_orders_iforest"))
anm_assigned.to_csv(REPORTS_DIR / "anomaly_assignments.csv", index=False)
anomaly_share = float((anm_assigned["Anomaly"] == 1).mean()) if "Anomaly" in anm_assigned.columns else None
track_rows.append({
    "track": "anomaly",
    "status": "ok",
    "primary_metric": "anomaly_share",
    "primary_metric_value": anomaly_share,
    "model_artifact": str(MODELS_DIR / "anomaly_orders_iforest.pkl"),
})

# Time series forecasting: daily orders
ts_exp = TSForecastingExperiment()
ts_ready = ts_df.sort_values("order_date").set_index("order_date").asfreq("D")
ts_ready["daily_orders"] = ts_ready["daily_orders"].fillna(0)
ts_exp.setup(data=ts_ready, target="daily_orders", fh=30, fold=3, session_id=SEED, verbose=False)
ts_model = ts_exp.compare_models(turbo=True)
ts_metrics = ts_exp.pull().copy()
ts_forecast = ts_exp.predict_model(ts_model, fh=30)
ts_exp.save_model(ts_model, str(MODELS_DIR / "timeseries_daily_orders_best"))
ts_forecast.to_csv(REPORTS_DIR / "timeseries_forecast.csv")
ts_metrics.to_csv(REPORTS_DIR / "timeseries_metrics.csv", index=False)
track_rows.append({
    "track": "time_series",
    "status": "ok",
    "primary_metric": "MASE",
    "primary_metric_value": float(ts_metrics.iloc[0]["MASE"]) if "MASE" in ts_metrics.columns else None,
    "model_artifact": str(MODELS_DIR / "timeseries_daily_orders_best"),
})

                                    Model  Accuracy     AUC  Recall   Prec.  \
lr                    Logistic Regression    0.9997  1.0000  0.9995  0.9998   
lightgbm  Light Gradient Boosting Machine    0.9989  1.0000  0.9990  0.9989   
rf               Random Forest Classifier    0.9988  0.9999  0.9987  0.9990   

              F1   Kappa     MCC  TT (Sec)  
lr        0.9997  0.9993  0.9993    0.7267  
lightgbm  0.9989  0.9979  0.9979    0.6933  
rf        0.9988  0.9977  0.9977    0.7900  
                 Model  Accuracy  AUC  Recall   Prec.      F1   Kappa     MCC
0  Logistic Regression    0.9998  1.0  0.9998  0.9999  0.9998  0.9997  0.9997
Transformation Pipeline and Model Successfully Saved


                                    Model     MAE        MSE     RMSE      R2  \
lr                      Linear Regression  0.0670     1.3127   1.1208  1.0000   
rf                Random Forest Regressor  0.5710   122.0346  10.4797  0.9973   
lightgbm  Light Gradient Boosting Machine  3.1662  1659.7882  40.4873  0.9635   

           RMSLE    MAPE  TT (Sec)  
lr        0.0058  0.0005    0.5633  
rf        0.0084  0.0018    0.9733  
lightgbm  0.0209  0.0106    0.1467  
               Model    MAE     MSE    RMSE   R2   RMSLE    MAPE
0  Linear Regression  0.058  1.1885  1.0902  1.0  0.0053  0.0005
Transformation Pipeline and Model Successfully Saved


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.7541          93931.566          0.5107            0           0   

   Completeness  
0             0  
Transformation Pipeline and Model Successfully Saved


Transformation Pipeline and Model Successfully Saved


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
naive,Naive Forecaster,2.9145,1.4619,87.1778,98.0006,99079191802150912.0000,0.6594,-1.5154,0.1933
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,3.1150,1.5563,93.7523,104.2157,122357608566246784.0000,0.6273,-2.2157,0.2433
stlf,STLF,3.2639,1.6529,98.3439,110.6750,120353160526507440.0000,0.6272,-2.7322,0.0267
ets,ETS,3.3029,1.6593,99.4657,111.1064,126438502617619264.0000,0.6323,-2.7206,0.0433
exp_smooth,Exponential Smoothing,3.3040,1.6594,99.5000,111.1143,126673193813444528.0000,0.6325,-2.7197,0.0633
theta,Theta Forecaster,3.3523,1.6698,100.6087,111.8756,125948654578280464.0000,0.6644,-2.4419,0.0233
arima,ARIMA,3.3931,1.7348,102.1283,116.1836,120382537072466880.0000,0.6434,-2.9116,0.0633
lightgbm_cds_dt,Light Gradient Boosting w/ Cond. Deseasonalize & Detrending,3.4237,1.7122,103.1311,114.6450,137221098021261520.0000,0.6351,-2.9749,0.2600
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,3.4329,1.7155,103.3818,114.8713,136446758389887296.0000,0.6407,-2.9276,0.4467
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,3.4805,1.7345,104.8431,116.1405,137770211546056592.0000,0.6418,-3.0333,0.2633


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,Naive Forecaster,0.0231,0.0126,0.7333,0.8563,3302639726738363.5000,1.4667,-2.7500


Transformation Pipeline and Model Successfully Saved


## 9) Consolidate Track Results
Create a single leaderboard file for downstream reporting and BI.

In [8]:
track_results = pd.DataFrame(track_rows)
track_results.to_csv(REPORTS_DIR / "pycaret_track_leaderboard.csv", index=False)
display(track_results)
print(f"Wrote {REPORTS_DIR / 'pycaret_track_leaderboard.csv'}")

,track,status,primary_metric,primary_metric_value,model_artifact
0,classification,ok,Accuracy,0.999700,c:\Users\sayee\Documents\olist-ai-analyst\mode...
1,regression,ok,MAE,0.067000,c:\Users\sayee\Documents\olist-ai-analyst\mode...
2,clustering,ok,Silhouette,0.754100,c:\Users\sayee\Documents\olist-ai-analyst\mode...
3,anomaly,ok,anomaly_share,0.050008,c:\Users\sayee\Documents\olist-ai-analyst\mode...
4,time_series,ok,MASE,2.914500,c:\Users\sayee\Documents\olist-ai-analyst\mode...


Wrote c:\Users\sayee\Documents\olist-ai-analyst\reports\pycaret_track_leaderboard.csv


## 10) Export BI-Ready Tables
Generate Power BI-friendly tables for model performance, segments, anomalies, forecasts, and scored orders.

In [9]:
BI_DIR = PROCESSED_DIR / "bi"
BI_DIR.mkdir(parents=True, exist_ok=True)

# 1) Model performance summary
bi_model_performance = track_results.copy()
bi_model_performance["run_timestamp_utc"] = pd.Timestamp.utcnow()
bi_model_performance.to_parquet(BI_DIR / "bi_model_performance.parquet", index=False)

# 2) Customer segments
bi_customer_segments = clu_assigned.copy().reset_index(drop=True)
cluster_col = "Cluster" if "Cluster" in bi_customer_segments.columns else None
if cluster_col is not None:
    bi_customer_segments = bi_customer_segments.rename(columns={cluster_col: "segment_id"})
bi_customer_segments["run_timestamp_utc"] = pd.Timestamp.utcnow()
bi_customer_segments.to_parquet(BI_DIR / "bi_customer_segments.parquet", index=False)

# 3) Anomaly monitor
bi_anomaly = anm_assigned.copy().reset_index(drop=True)
anomaly_col = "Anomaly" if "Anomaly" in bi_anomaly.columns else None
score_col = "Anomaly_Score" if "Anomaly_Score" in bi_anomaly.columns else None
if anomaly_col is not None:
    bi_anomaly = bi_anomaly.rename(columns={anomaly_col: "is_anomaly"})
if score_col is not None:
    bi_anomaly = bi_anomaly.rename(columns={score_col: "anomaly_score"})
bi_anomaly["run_timestamp_utc"] = pd.Timestamp.utcnow()
bi_anomaly.to_parquet(BI_DIR / "bi_anomaly_monitor.parquet", index=False)

# 4) Forecast trends
bi_forecast = ts_forecast.reset_index().copy()
if "index" in bi_forecast.columns and "forecast_date" not in bi_forecast.columns:
    bi_forecast = bi_forecast.rename(columns={"index": "forecast_date"})
if "y_pred" in bi_forecast.columns and "forecast_value" not in bi_forecast.columns:
    bi_forecast = bi_forecast.rename(columns={"y_pred": "forecast_value"})
bi_forecast["run_timestamp_utc"] = pd.Timestamp.utcnow()
bi_forecast.to_parquet(BI_DIR / "bi_forecast_trends.parquet", index=False)

# 5) Scored orders (classification + regression)
bi_scored_orders = reg_pred.copy().reset_index(drop=True)
if "prediction_label" in bi_scored_orders.columns:
    bi_scored_orders = bi_scored_orders.rename(columns={"prediction_label": "pred_total_payment"})
elif "Label" in bi_scored_orders.columns:
    bi_scored_orders = bi_scored_orders.rename(columns={"Label": "pred_total_payment"})

cls_cols = [c for c in ["prediction_label", "Label", "Score", "prediction_score"] if c in cls_pred.columns]
for c in cls_cols:
    out_name = {
        "prediction_label": "pred_high_value_order",
        "Label": "pred_high_value_order",
        "Score": "pred_high_value_score",
        "prediction_score": "pred_high_value_score",
    }[c]
    bi_scored_orders[out_name] = cls_pred[c].reset_index(drop=True)

bi_scored_orders["run_timestamp_utc"] = pd.Timestamp.utcnow()
bi_scored_orders.to_parquet(BI_DIR / "bi_scored_orders.parquet", index=False)

produced = sorted(p.name for p in BI_DIR.glob("*.parquet"))
print("BI tables written:")
for name in produced:
    print("-", name)

display(pd.DataFrame({"table": produced}))

BI tables written:
- bi_anomaly_monitor.parquet
- bi_customer_segments.parquet
- bi_forecast_trends.parquet
- bi_model_performance.parquet
- bi_scored_orders.parquet


,table
0,bi_anomaly_monitor.parquet
1,bi_customer_segments.parquet
2,bi_forecast_trends.parquet
3,bi_model_performance.parquet
4,bi_scored_orders.parquet


## 11) Export BI Data Dictionary
Create a column-level reference file for Power BI semantic model setup.

In [10]:
data_dictionary_rows = []

for parquet_path in sorted(BI_DIR.glob("*.parquet")):
    df_tmp = pd.read_parquet(parquet_path)
    for col in df_tmp.columns:
        data_dictionary_rows.append({
            "table_name": parquet_path.stem,
            "column_name": col,
            "dtype": str(df_tmp[col].dtype),
            "sample_non_null": None if df_tmp[col].dropna().empty else str(df_tmp[col].dropna().iloc[0]),
        })

bi_dictionary = pd.DataFrame(data_dictionary_rows).sort_values(["table_name", "column_name"])
bi_dictionary_path = REPORTS_DIR / "bi_data_dictionary.csv"
bi_dictionary.to_csv(bi_dictionary_path, index=False)

print(f"Wrote {bi_dictionary_path}")
display(bi_dictionary.head(20))

Wrote c:\Users\sayee\Documents\olist-ai-analyst\reports\bi_data_dictionary.csv


,table_name,column_name,dtype,sample_non_null
5,bi_anomaly_monitor,anomaly_score,float64,-0.19898981639455338
4,bi_anomaly_monitor,is_anomaly,int32,0
0,bi_anomaly_monitor,item_count,float32,1.0
6,bi_anomaly_monitor,run_timestamp_utc,"datetime64[us, UTC]",2026-03-14 06:46:06.536253+00:00
2,bi_anomaly_monitor,total_freight,float32,8.72
3,bi_anomaly_monitor,total_payment,float32,38.71
1,bi_anomaly_monitor,total_price,float32,29.99
9,bi_customer_segments,avg_order_value,float32,141.9
11,bi_customer_segments,run_timestamp_utc,"datetime64[us, UTC]",2026-03-14 06:46:06.501463+00:00
10,bi_customer_segments,segment_id,object,Cluster 0
